## User-Based + Item-Based

![ModelPredict](ModelPredict.png)

In [24]:
import ast
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from rapidfuzz import process, fuzz

### Load Steam Dataset

In [25]:
# Подгрузим Steam Dataset, возьмем только записи тех кто уже поиграл в игру.

steam_raw = pd.read_csv('../SteamDataset200K/steam-200k.csv', header=None, names=['user_id', 'game', 'behavior', 'hours', 'zero'])

df = steam_raw[steam_raw['behavior'] == 'play'].copy()
df = df[['user_id', 'game', 'hours']].reset_index(drop=True)

df.head()

,user_id,game,hours
0,151603712,The Elder Scrolls V Skyrim,273.0
1,151603712,Fallout 4,87.0
2,151603712,Spore,14.9
3,151603712,Fallout New Vegas,12.1
4,151603712,Left 4 Dead 2,8.9


In [26]:
# Конвертируем игровые часы в рейтинг, чтобы метрика стала более читаемой.
# Масштабируем логарифмически.

def hours_to_rating(hours_series):
    log_h = np.log1p(hours_series)
    scaler = MinMaxScaler(feature_range=(1, 10))
    ratings = scaler.fit_transform(log_h.values.reshape(-1, 1)).flatten()

    return np.round(ratings, 2)


df['rating'] = hours_to_rating(df['hours'])

df.head(10)

,user_id,game,hours,rating
0,151603712,The Elder Scrolls V Skyrim,273.0,6.35
1,151603712,Fallout 4,87.0,5.25
2,151603712,Spore,14.9,3.59
3,151603712,Fallout New Vegas,12.1,3.40
4,151603712,Left 4 Dead 2,8.9,3.13
5,151603712,HuniePop,8.5,3.09
6,151603712,Path of Exile,8.1,3.05
7,151603712,Poly Bridge,7.5,2.98
8,151603712,Left 4 Dead,3.3,2.32
9,151603712,Team Fortress 2,2.8,2.20


In [27]:
# Делаем датасет менее разряженным (это может повлечь проблемы: непопулярные игры не будут рекомендоваться, но мы подумаем над этим позже)

MIN_GAMES_PER_USER = 3
MIN_USERS_PER_GAME = 5

user_counts = df.groupby('user_id')['game'].count()
game_counts = df.groupby('game')['user_id'].count()

active_users = user_counts[user_counts >= MIN_GAMES_PER_USER].index
popular_games = game_counts[game_counts >= MIN_USERS_PER_GAME].index

df_filtered = df[df['user_id'].isin(active_users) & df['game'].isin(popular_games)]

user_item_matrix = df_filtered.pivot_table(
    index='user_id',
    columns='game',
    values='rating',
    aggfunc='mean'
).fillna(0)

user_item_matrix

game,1... 2... 3... KICK IT! (Drop That Beat Like an Ugly Baby),100% Orange Juice,12 Labours of Hercules,12 Labours of Hercules II The Cretan Bull,140,3DMark,404Sight,60 Seconds!,7 Days to Die,8BitBoy,...,Zeno Clash,Zombie Army Trilogy,Zombie Driver,Zombie Panic Source,Zombies Monsters Robots,ibb & obb,resident evil 4 / biohazard 4,sZone-Online,the static speaks my name,theHunter
user_id,,,,,,,,,,,,,,,,,,,,,
5250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
76767,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
86540,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
229911,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
298950,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.55,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306547522,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
306971738,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
308695132,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### User-Based

In [28]:
def user_based_rec(user_id, user_item_matrix, n_neighbors=20, top_n=20):
    if user_id not in user_item_matrix.index:
        return None

    sim_matrix = cosine_similarity(user_item_matrix.values)
    sim_df = pd.DataFrame(sim_matrix, index=user_item_matrix.index, columns=user_item_matrix.index)

    user_sim = sim_df[user_id].drop(user_id).sort_values(ascending=False)
    neighbors = user_sim.head(n_neighbors)

    user_games = user_item_matrix.loc[user_id]
    unplayed = user_games[user_games == 0].index

    neighbor_ratings = user_item_matrix.loc[neighbors.index, unplayed]
    weights = neighbors.values.reshape(-1, 1)

    weighted_sum = (neighbor_ratings * weights).sum(axis=0)
    weight_total = (neighbor_ratings > 0).astype(int).T.dot(weights.flatten())
    weight_total = weight_total.replace(0, np.nan)

    predicted = (weighted_sum / weight_total).dropna().sort_values(ascending=False)

    result = predicted.head(top_n).reset_index()
    result.columns = ['game', 'ub_score']

    return result


sample_user = user_item_matrix.index[0]
games_sample_user = user_item_matrix.loc[sample_user]
played = games_sample_user[games_sample_user != 0]
ub_recs = user_based_rec(sample_user, user_item_matrix)

played, ub_recs

(game
 Alien Swarm                 2.63
 Cities Skylines             5.74
 Deus Ex Human Revolution    4.93
 Dota 2                      1.08
 Portal 2                    3.51
 Team Fortress 2             1.48
 Name: 5250, dtype: float64,
                                            game  ub_score
 0                         Counter-Strike Source  6.550000
 1                          Farming Simulator 15  5.250000
 2                          Realm of the Mad God  5.110000
 3                                Clicker Heroes  5.020000
 4                            Half-Life 2 Update  4.970000
 5                                   Garry's Mod  4.960000
 6                            Tabletop Simulator  4.860000
 7               Never Alone (Kisima Ingitchuna)  4.810000
 8                                    Watch_Dogs  4.780000
 9   Grand Theft Auto Episodes from Liberty City  4.530000
 10                                  Train Fever  4.490000
 11      Call of Duty Black Ops II - Multiplayer  4.4

### Load PlayMyData

In [29]:
pc = pd.read_csv('../PlayMyData/all_games_PC.csv')
ps = pd.read_csv('../PlayMyData/all_games_PlayStation.csv')
genres_df = pd.read_csv('../PlayMyData/genres.csv')

games_df = pd.concat([pc, ps], ignore_index=True).drop_duplicates(subset='id').reset_index(drop=True)

games_df['rating'] = pd.to_numeric(games_df['rating'], errors='coerce')

games_df.head()

,genres,id,name,platforms,summary,storyline,rating,main,extra,completionist,review_score,review_count,people_polled
0,[5],274203,Short 'n Quick,[6],this is a techstyled map taking place in a war...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,[5],256819,Caverns of Darkness,[6],The final rift was closed and the Hell War was...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"[26, 31]",232860,"Nu, pogodi! Vypusk 1: Pogonya",[6],The wolf decides to take revenge on the Hare f...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"[12, 15, 16]",228979,Power Dolls 5,[6],The fifth entry in Kogado Studios allfemale me...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,[2],225648,Kapsyljakt med Anki & Pytte,[6],A pointandclick game based on the Swedish chil...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
def parse_genres(value):
    if not isinstance(value, str):
        return []
    value = value.strip()
    if value == '' or value == 'Missing':
        return []
    try:
        result = ast.literal_eval(value)

        return result if isinstance(result, list) else []
    except Exception:
        return []


games_df['genres_list'] = games_df['genres'].apply(parse_genres)

#One-hot encoding
all_genre_ids = genres_df['genre_id'].tolist()
for gid in all_genre_ids:
    games_df[f'genre_{gid}'] = games_df['genres_list'].apply(lambda x: int(gid in x))

num_features = ['rating', 'review_score', 'main', 'extra', 'completionist']
for column in num_features:
    games_df[column] = pd.to_numeric(games_df[column], errors='coerce')
    games_df[column] = games_df[column].fillna(games_df[column].median())
    max_value = games_df[column].max()

    if max_value > 0:
        games_df[column] = games_df[column] / max_value

games_df.head(10)

,genres,id,name,platforms,summary,storyline,rating,main,extra,completionist,...,genre_26,genre_25,genre_30,genre_31,genre_33,genre_34,genre_32,genre_35,genre_36,genre_2
0,[5],274203,Short 'n Quick,[6],this is a techstyled map taking place in a war...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,[5],256819,Caverns of Darkness,[6],The final rift was closed and the Hell War was...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,"[26, 31]",232860,"Nu, pogodi! Vypusk 1: Pogonya",[6],The wolf decides to take revenge on the Hare f...,Missing,0.7,0.000309,0.0,0.0,...,1,0,0,1,0,0,0,0,0,0
3,"[12, 15, 16]",228979,Power Dolls 5,[6],The fifth entry in Kogado Studios allfemale me...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,[2],225648,Kapsyljakt med Anki & Pytte,[6],A pointandclick game based on the Swedish chil...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,1
5,[10],204650,Rage Rally,[6],Missing,Missing,0.7,0.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
6,[34],194713,Ripple: Blue Seal he Youkoso,[6],Main character loses a job and applies for an ...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,1,0,0,0,0
7,[31],73345,The Mystery of the Nautilus,[6],This adventure game was inspired by the Jules ...,The story begins aboard the USS Shark a resear...,0.7,0.000309,0.0,0.0,...,0,0,0,1,0,0,0,0,0,0
8,"[2, 31]",71917,JumpStart: Animal Adventures,"[6, 14]",Educational game for teaching kids about wild ...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,1,0,0,0,0,0,1
9,Missing,61935,Farland Symphony,[6],A spinoff title in the Farland Story RPG franc...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [31]:
# Решаем проблему исключения нишевых игр
# Для Item-Based расширяем матрицу сходства, игры из Steam датасета используем как сиды (игры юзера), а при нахождении похожих игр, берем все из PlayMyData (85k).
# Это позволит рекомендовать нишевые игры, даже если их вообще не было в датасете Steam.

steam_games_names = set(user_item_matrix.columns)

genre_columns = [f'genre_{gid}' for gid in all_genre_ids]
num_columns = ['rating', 'review_score', 'main', 'extra', 'completionist']
feature_columns = genre_columns + num_columns

GENRE_WEIGHT = 3.0
NUMERIC_WEIGHT = 1.5

all_content_games = games_df.copy().drop_duplicates(subset='name').set_index('name')
feature_matrix_all = all_content_games[feature_columns].fillna(0).values.copy().astype(float)

n_genre = len(genre_columns)
feature_matrix_all[:, :n_genre] *= GENRE_WEIGHT
feature_matrix_all[:, :n_genre] *= NUMERIC_WEIGHT

steam_in_playmydata = [g for g in steam_games_names if g in all_content_games.index]
steam_index = [all_content_games.index.get_loc(g) for g in steam_in_playmydata]
steam_vectors = feature_matrix_all[steam_index]

item_sim_matrix = cosine_similarity(feature_matrix_all, steam_vectors)
item_sim_df = pd.DataFrame(
    item_sim_matrix,
    index=all_content_games.index,
    columns=steam_in_playmydata
)

item_sim_df

,NBA 2K16,Primal Carnage,Zeno Clash,Betrayer,McPixel,Blades of Time,Mass Effect,Abyss Odyssey,Sacred Citadel,Infinite Crisis,...,Spiral Knights,March of the Eagles,And Yet It Moves,Alan Wake,LEGO The Hobbit,Fuse,Legend of Dungeon,Audition Online,I am Bread,Lost Planet 2
name,,,,,,,,,,,,,,,,,,,,,
Short 'n Quick,0.043988,0.714946,0.509201,0.584949,0.019901,0.507088,0.717459,0.022048,0.023212,0.019365,...,0.017963,0.020132,0.023688,0.716725,0.024123,0.714151,0.020794,0.039686,0.014378,0.714372
Caverns of Darkness,0.043988,0.714946,0.509201,0.584949,0.019901,0.507088,0.717459,0.022048,0.023212,0.019365,...,0.017963,0.020132,0.023688,0.716725,0.024123,0.714151,0.020794,0.039686,0.014378,0.714372
"Nu, pogodi! Vypusk 1: Pogonya",0.031401,0.020829,0.015586,0.417572,0.362503,0.361990,0.025635,0.417391,0.016570,0.013824,...,0.361408,0.014371,0.016910,0.511641,0.418356,0.509804,0.416826,0.028330,0.322570,0.509962
Power Dolls 5,0.025721,0.017061,0.012767,0.013170,0.011637,0.296514,0.419525,0.012892,0.342365,0.582081,...,0.296037,0.341027,0.013851,0.019709,0.342684,0.016207,0.341431,0.023206,0.008407,0.016404
Kapsyljakt med Anki & Pytte,0.043988,0.029178,0.021834,0.022523,0.507806,0.018991,0.035910,0.022048,0.023212,0.019365,...,0.017963,0.020132,0.023688,0.033706,0.024123,0.027716,0.020794,0.039686,0.014378,0.028054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Magical Drop 3,0.057487,0.037512,0.028407,0.029705,0.507070,0.024917,0.046959,0.028906,0.030217,0.025059,...,0.023631,0.026863,0.585199,0.044036,0.585841,0.036341,0.026759,0.050573,0.450354,0.036616
Tenchu 2: Birth of the Stealth Assassins,0.055577,0.036388,0.027487,0.584664,0.507326,0.506592,0.045390,0.584148,0.029235,0.024275,...,0.505555,0.025856,0.029748,0.717793,0.586020,0.713704,0.582634,0.049151,0.450685,0.713889
Dragon Warrior VII,0.033996,0.022756,0.016917,0.418052,0.363083,0.710194,0.513355,0.417901,0.418616,0.362787,...,0.709683,0.015375,0.018385,0.512712,0.819383,0.510473,0.818653,0.031100,0.322816,0.510706


In [32]:
FUZZY_THRESHOLD = 93

BLACKLIST = {'Besiege', 'Jamestown', 'Color Symphony', 'Goat Simulator', 'Borderlands 2 RU', 'Ragnarok', 'Knights of Pen and Paper +1'}

playmydata_names = all_content_games.index.tolist()
unmatched = steam_games_names - set(all_content_games.index)

fuzzy_map = {}
for steam_name in unmatched:
    if steam_name in BLACKLIST:
        continue

    match, score, _ = process.extractOne(steam_name,
                                         playmydata_names,
                                         scorer=fuzz.token_sort_ratio)
    if score >= FUZZY_THRESHOLD:
        fuzzy_map[steam_name] = match

fuzzy_rows = []
for steam_name, playmydata_name in fuzzy_map.items():
    if playmydata_name in all_content_games.index:
        row = all_content_games.loc[playmydata_name].copy()
        row.name = steam_name
        fuzzy_rows.append(row)

if fuzzy_rows:
    fuzzy_df = pd.DataFrame(fuzzy_rows)
    all_content_games = pd.concat([all_content_games, fuzzy_df])

len(all_content_games)

84030

### Item-Based

In [38]:
def item_based_rec(user_id, user_item_matrix, item_sim_df, n_neighbors=20, top_n=20, sim_threshold=0.7, min_seeds=2):
    if user_id not in user_item_matrix.index:
        return None

    user_games = user_item_matrix.loc[user_id]
    played = user_games[user_games != 0]
    played_known = played[played.index.isin(item_sim_df.columns)]

    if len(played_known) == 0:
        return None

    already_played = set(played.index)
    candidates = [g for g in item_sim_df.index if g not in already_played]

    sim_block = item_sim_df.loc[candidates, played_known.index].values.copy()

    for i in range(len(sim_block)):
        row = sim_block[i]
        threshold_idx = np.argsort(row)[:-n_neighbors]
        row[threshold_idx] = 0

    seeds_above_threshold = (sim_block >= sim_threshold).sum(axis=1)
    mask = seeds_above_threshold >= min_seeds

    sim_block = sim_block[mask]
    filtered_candidates = [c for c, m in zip(candidates, mask) if m]

    ratings = played_known.values
    weighted_sum = sim_block.dot(ratings)
    weight_total = sim_block.sum(axis=1)
    weight_total[weight_total == 0] = np.nan

    result = pd.Series(weighted_sum, index=filtered_candidates).dropna()
    result = result.sort_values(ascending=False).head(top_n).reset_index()
    result.columns = ['game', 'ib_score']

    return result


sample_user = user_item_matrix.index[0]
games_sample_user = user_item_matrix.loc[sample_user]
played = games_sample_user[games_sample_user != 0]

ib_recs = item_based_rec(sample_user, user_item_matrix, item_sim_df)

played, ib_recs

(game
 Alien Swarm                 2.63
 Cities Skylines             5.74
 Deus Ex Human Revolution    4.93
 Dota 2                      1.08
 Portal 2                    3.51
 Team Fortress 2             1.48
 Name: 5250, dtype: float64,
                                                game  ib_score
 0                 Halo: The Master Chief Collection  5.958774
 1                                   Resident Evil 4  5.958244
 2                                       Half-Life 2  5.958231
 3                                    Counter-Strike  5.958055
 4                                      Doom Eternal  5.957872
 5                                 The Ultimate Doom  5.957798
 6                                 Unreal Tournament  5.957736
 7                                          Returnal  5.957657
 8   No One Lives Forever 2: A Spy in H.A.R.M.'s Way  5.957594
 9                                            Halo 3  5.957589
 10         Medal of Honor: Allied Assault War Chest  5.957574
 11  

### Final Model (User-Based + Item-Based)

In [42]:
# alpha - вес User-Based оценки
# beta - вес Item-Based оценки
# boost_factor - множитель для игр из пересечения User-Based и Item-Based
# discount_factor - множитель для игр только из одного

def hybrid_rec(user_id, user_item_matrix, item_sim_df, n_neighbours=20, top_n=20,
               alpha=0.8, beta=0.2, boost_factor=1.3, discount_factor=0.7):
    ub = user_based_rec(user_id, user_item_matrix, n_neighbours, top_n)
    ib = item_based_rec(user_id, user_item_matrix, item_sim_df, n_neighbours, top_n)

    if ub is None or ib is None:
        return None

    def normalize(series):
        mn, mx = series.min(), series.max()

        return (series - mn) / (mx - mn + 1e-9)

    ub['ub_score'] = normalize(ub['ub_score'])
    ib['ib_score'] = normalize(ib['ib_score'])

    merged = pd.merge(ub, ib, on='game', how='outer')

    both = merged.dropna(subset=['ub_score', 'ib_score']).copy()
    both['final_score'] = (alpha * both['ub_score'] + beta * both['ib_score']) * boost_factor

    only_ub = merged[merged['ib_score'].isna()].copy()
    only_ub['final_score'] = only_ub['ub_score'] * discount_factor

    only_ib = merged[merged['ub_score'].isna()].copy()
    only_ib['final_score'] = only_ib['ib_score'] * discount_factor

    result = pd.concat([both, only_ub, only_ib], ignore_index=True)
    result = result[['game', 'final_score']].sort_values('final_score', ascending=False).head(top_n).reset_index(drop=True)

    result.index += 1

    return result


sample_user = user_item_matrix.index[0]
games_sample_user = user_item_matrix.loc[sample_user]
played = games_sample_user[games_sample_user != 0]

hybrid_recs = hybrid_rec(sample_user, user_item_matrix, item_sim_df)

played, hybrid_recs

(game
 Alien Swarm                 2.63
 Cities Skylines             5.74
 Deus Ex Human Revolution    4.93
 Dota 2                      1.08
 Portal 2                    3.51
 Team Fortress 2             1.48
 Name: 5250, dtype: float64,
                                                game  final_score
 1                             Counter-Strike Source     0.700000
 2                 Halo: The Master Chief Collection     0.700000
 3                                   Resident Evil 4     0.478233
 4                                       Half-Life 2     0.472737
 5                                    Counter-Strike     0.399179
 6                              Farming Simulator 15     0.373835
 7                              Realm of the Mad God     0.338710
 8                                      Doom Eternal     0.322538
 9                                    Clicker Heroes     0.316129
 10                               Half-Life 2 Update     0.303584
 11                                

In [50]:
def evaluate(user_item_matrix, item_sim_df, rec_func, test_ratio=0.2, top_n=10, n_users=100, test_users_override=None, **kwargs):
    if test_users_override is not None:
        eligible = (user_item_matrix > 0).sum(axis=1)
        test_users = [u for u in test_users_override if eligible.get(u, 0) >= 2]
    else:
        eligible = (user_item_matrix > 0).sum(axis=1)
        eligible = eligible[eligible >= 10].index
        test_users = np.random.choice(eligible,
                                      size=min(n_users, len(eligible)),
                                      replace=False)

    metrics = {'precision': [], 'recall': [], 'hit_rate': []}

    for uid in test_users:
        played = user_item_matrix.loc[uid]
        played = played[played > 0]

        if len(played) < 0:
            continue

        n_test = max(1, int(len(played) * test_ratio))
        test_games = np.random.choice(played.index, size=n_test, replace=False)

        temp = user_item_matrix.copy()
        temp.loc[uid, test_games] = 0

        try:
            if rec_func in (hybrid_rec, item_based_rec):
                recs = rec_func(uid, temp, item_sim_df, top_n=top_n * 2, **kwargs)
            else:
                recs = rec_func(uid, temp, top_n=top_n * 2, **kwargs)

            if recs is None or len(recs) == 0:
                continue

            rec_games = recs['game'].tolist()[:top_n]
        except Exception:
            continue

        relevant = set(test_games)
        hits = [1 if g in relevant else 0 for g in rec_games]

        precision = sum(hits) / top_n
        recall = sum(hits) / len(relevant)
        hit_rate = int(sum(hits) > 0)

        metrics['precision'].append(precision)
        metrics['recall'].append(recall)
        metrics['hit_rate'].append(hit_rate)

    results = {k: round(np.mean(v), 4) for k, v in metrics.items()}
    results['n_evaluated'] = len(metrics['precision'])

    return results


np.random.seed(42)
result_ub = evaluate(user_item_matrix, item_sim_df, user_based_rec)
result_ib = evaluate(user_item_matrix, item_sim_df, item_based_rec)
result_hybrid = evaluate(user_item_matrix, item_sim_df, hybrid_rec)

metrics_df = pd.DataFrame({
    'User-Based': result_ub,
    'Item-Based': result_ib,
    'Hybrid': result_hybrid
}).T

metrics_df

,precision,recall,hit_rate,n_evaluated
User-Based,0.0360,0.0778,0.2900,100.0
Item-Based,0.0053,0.0069,0.0532,94.0
Hybrid,0.0180,0.0438,0.1800,100.0


In [41]:
params_grid = [
    {'alpha': 0.7, 'beta': 0.3, 'boost_factor': 1.2, 'discount_factor': 0.7},
    {'alpha': 0.6, 'beta': 0.4, 'boost_factor': 1.3, 'discount_factor': 0.8},
    {'alpha': 0.5, 'beta': 0.5, 'boost_factor': 1.5, 'discount_factor': 0.9},
    {'alpha': 0.4, 'beta': 0.6, 'boost_factor': 1.3, 'discount_factor': 0.8},
    {'alpha': 0.8, 'beta': 0.2, 'boost_factor': 1.3, 'discount_factor': 0.7},
    {'alpha': 0.8, 'beta': 0.2, 'boost_factor': 1.3, 'discount_factor': 0.8},
]

rows = []
for params in params_grid:
    rec = evaluate(user_item_matrix, item_sim_df, hybrid_rec, **params)
    rows.append({**params, **rec})

tuning_df = pd.DataFrame(rows)

tuning_df

,alpha,beta,boost_factor,discount_factor,precision,recall,hit_rate,n_evaluated
0,0.7,0.3,1.2,0.7,0.0150,0.0435,0.1300,100
1,0.6,0.4,1.3,0.8,0.0150,0.0192,0.1200,100
2,0.5,0.5,1.5,0.9,0.0152,0.0390,0.1414,99
3,0.4,0.6,1.3,0.8,0.0170,0.0400,0.1600,100
4,0.8,0.2,1.3,0.7,0.0230,0.0661,0.2100,100
5,0.8,0.2,1.3,0.8,0.0170,0.0460,0.1600,100


In [51]:
# Исследование 1. Расчет на активных и не активных пользователях

game_counts = (user_item_matrix > 0).sum(axis=1).sort_values()
n = len(game_counts)

inactive_users = game_counts.iloc[:int(n * 0.1)].index
active_users = game_counts.iloc[int(n * 0.9):].index

results_active = evaluate(user_item_matrix, item_sim_df, hybrid_rec, test_users_override=active_users)
results_inactive = evaluate(user_item_matrix, item_sim_df, hybrid_rec, test_users_override=inactive_users, test_ratio=0.34)

results_active, results_inactive

({'precision': np.float64(0.0242),
  'recall': np.float64(0.0204),
  'hit_rate': np.float64(0.2161),
  'n_evaluated': 347},
 {'precision': np.float64(0.0172),
  'recall': np.float64(0.1718),
  'hit_rate': np.float64(0.1718),
  'n_evaluated': 262})

In [53]:
# Исследование 2. Холодный старт

def cold_start_recommendations(liked_games, item_sim_df):
    fake_user_id = 'cold_start_user'

    fake_row = pd.Series(0.0, index=user_item_matrix.columns)
    known_games = [g for g in liked_games if g in user_item_matrix.columns]
    fake_row[known_games] = 5.0

    temp_matrix = pd.concat([
        user_item_matrix,
        fake_row.to_frame(fake_user_id).T
    ])

    return item_based_rec(fake_user_id, temp_matrix, item_sim_df)


liked = ['Half-Life 2', 'Portal 2', 'Dota 2']
recs = cold_start_recommendations(liked, item_sim_df)
print(recs.head(10))

                                      game  ib_score
0                           The Last of Us  7.347238
1                The Last of Us Remastered  7.346491
2                           The Orange Box  7.342426
3                   The Last of Us Part II  7.337617
4               Uncharted 4: A Thief's End  7.337287
5               Metal Gear Solid: Integral  7.336935
6        Resident Evil: Origins Collection  7.336419
7                             Alan Wake II  7.334233
8  Uncharted: Legacy of Thieves Collection  7.333789
9            Metal Gear Solid 2: Substance  7.330922


In [54]:
# Оценка холодного старта

def evaluate_cold_start(user_item_matrix, item_sim_df, n_bootstrap=3, top_n=20, n_users=100):
    eligible = (user_item_matrix > 0).sum(axis=1)
    eligible = eligible[eligible >= 10].index
    test_users = np.random.choice(eligible,
                                  size=min(n_users, len(eligible)),
                                  replace=False)

    metrics = {'precision': [], 'recall': [], 'hit_rate': []}

    for uid in test_users:
        played = user_item_matrix.loc[uid]
        played = played[played > 0]

        bootstrap = list(np.random.choice(played.index,
                                          size=min(n_bootstrap, len(played)),
                                          replace=False))
        test_games = set(played.index) - set(bootstrap)

        if len(test_games) == 0:
            continue

        recs = cold_start_recommendations(bootstrap, item_sim_df)
        if recs is None or len(recs) == 0:
            continue

        rec_games = recs['game'].tolist()[:top_n]
        hits      = [1 if g in test_games else 0 for g in rec_games]

        metrics['precision'].append(sum(hits) / top_n)
        metrics['recall'].append(sum(hits) / len(test_games))
        metrics['hit_rate'].append(int(sum(hits) > 0))

    return {k: round(np.mean(v), 4) for k, v in metrics.items()}


np.random.seed(42)
result_cold = evaluate_cold_start(user_item_matrix, item_sim_df)

result_cold

{'precision': np.float64(0.0032),
 'recall': np.float64(0.0044),
 'hit_rate': np.float64(0.0638)}